### Build a Simple LLM Application with LCEL
In this quickstart we'll show you how to build a simple LLM application with LangChain. This application will translate text from English into another language. This is a relatively simple LLM application - it's just a single LLM call plus some prompting. Still, this is a great way to get started with LangChain - a lot of features can be built with just some prompting and an LLM call!

After seeing this video, you'll have a high level overview of:

- Using language models

- Using PromptTemplates and OutputParsers

- Using LangChain Expression Language (LCEL) to chain components together

- Debugging and tracing your application using LangSmith

- Deploying your application with LangServe

In [24]:
### Open AI API Key and Open Source models--Llama3,Gemma2,mistral--Groq

import os
from dotenv import load_dotenv
load_dotenv()

import openai
openai.api_key=os.getenv("OPENAI_API_KEY")

groq_api_key=os.getenv("GROQ_API_KEY")

In [9]:
from langchain_openai import ChatOpenAI
from langchain_groq import ChatGroq
model=ChatGroq(model="openai/gpt-oss-20b",groq_api_key=groq_api_key)
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.6', 'langchain': '1.4.3'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x115a64640>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x115a64fc0>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [10]:
from langchain_core.messages import HumanMessage,SystemMessage

messages=[
    SystemMessage(content="Translate the following from English to French"),
    HumanMessage(content="Hello How are you?")
]

result=model.invoke(messages)

In [11]:
result

AIMessage(content='Bonjour, comment ça va?', additional_kwargs={'reasoning_content': 'We need to translate "Hello How are you?" to French. The instruction: "Translate the following from English to French". So answer: "Bonjour, comment ça va?" or "Bonjour, comment vas-tu?" But the original "Hello How are you?" is informal. So "Bonjour, comment vas-tu?" Or "Bonjour, comment ça va?" The best translation: "Bonjour, comment ça va ?" So respond with that.'}, response_metadata={'token_usage': {'completion_tokens': 103, 'prompt_tokens': 86, 'total_tokens': 189, 'completion_time': 0.106180011, 'completion_tokens_details': {'reasoning_tokens': 88}, 'prompt_time': 0.00410834, 'prompt_tokens_details': None, 'queue_time': 0.338506873, 'total_time': 0.110288351}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_639a5351c8', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a102e8-788d-7bb2-ba76-151781ee9ed8-0', tool_calls=[]

In [12]:
from langchain_core.output_parsers import StrOutputParser
parser=StrOutputParser()
parser.invoke(result)

'Bonjour, comment ça va?'

In [13]:
### Using LCEL- chain the components
chain=model|parser
chain.invoke(messages)

'Bonjour, comment vas‑tu?'

In [14]:
### Prompt Templates
from langchain_core.prompts import ChatPromptTemplate

generic_template="Trnaslate the following into {language}:"

prompt=ChatPromptTemplate.from_messages(
    [("system",generic_template),("user","{text}")]
)



In [15]:
result=prompt.invoke({"language":"French","text":"Hello"})

In [16]:
result

ChatPromptValue(messages=[SystemMessage(content='Trnaslate the following into French:', additional_kwargs={}, response_metadata={}), HumanMessage(content='Hello', additional_kwargs={}, response_metadata={})])

In [17]:
result.to_messages()

[SystemMessage(content='Trnaslate the following into French:', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Hello', additional_kwargs={}, response_metadata={})]

In [22]:
model.invoke(result)

AIMessage(content='Bonjour', additional_kwargs={'reasoning_content': 'The user says: "Hello". They want to translate the following into French: "Hello". So answer: "Bonjour".'}, response_metadata={'token_usage': {'completion_tokens': 36, 'prompt_tokens': 83, 'total_tokens': 119, 'completion_time': 0.039623013, 'completion_tokens_details': {'reasoning_tokens': 26}, 'prompt_time': 0.004844108, 'prompt_tokens_details': None, 'queue_time': 0.332338077, 'total_time': 0.044467121}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_66f3850a1a', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a1063e-e24f-7061-aaf3-634e58e0d67b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 83, 'output_tokens': 36, 'total_tokens': 119, 'output_token_details': {'reasoning': 26}})

In [23]:
##Chaining together components with LCEL
chain=prompt|model|parser
chain.invoke({"language":"French","text":"Hello"})

'Bonjour'